In [1]:
import asyncio
import websockets
import json
from IPython.display import clear_output

async def live_dashboard():
    uri = "ws://localhost:8765"
    
    # 🔄 OTO-RECONNECT DÖNGÜSÜ: Sunucu kapansa bile dashboard pes etmez
    while True:
        try:
            async with websockets.connect(uri) as websocket:
                # Başarılı bağlantı sağlandığında iç döngüye girer
                while True:
                    # Sunucudan güncel verileri talep et
                    await websocket.send(json.dumps({"action": "get_dashboard_data"}))
                    raw_data = await websocket.recv()
                    data = json.loads(raw_data)

                    # Ekranı temizle (Canlı panel hissi için)
                    clear_output(wait=True)

                    print(f"--- 📊 CANLI SINIF DURUMU ({data.get('active_students_count', 0)} Öğrenci) ---")
                    
                    for sid, info in data.get("students", {}).items():
                        status = info['state']
                        time_left = info.get('time_left_formatted', '00:00')
                        ders = info.get('exam_id', 'Bilinmiyor')
                        score = info.get('risk_score', 0)
                        level = info.get('risk_level', 'TEMİZ')

                        # Durum metni ve ikonu belirleme
                        if status == 'in_progress':
                            durum_ikonu = "🟢 Devam Ediyor"
                        elif status == 'disconnected_paused':
                            durum_ikonu = "🔴 Koptu (Donduruldu)"
                        elif status == 'completed':
                            durum_ikonu = "✅ Sınav Bitti"
                        else:
                            durum_ikonu = "⚠️ İhlal (Donduruldu)"

                        # Risk skoruna göre dinamik ikon belirleme
                        risk_icon = "🟢" if score < 40 else "🟠" if score < 80 else "🔴"

                        # Tek satırda öğrenci özeti
                        print(f"[{risk_icon} %{score:02d} | {level}] Öğrenci: {sid} | 📝 Sınav: {ders} | Süre: {time_left} | Durum: {durum_ikonu}")

                    print("-------------------------------------------------------")
                    # 2 saniyede bir güncelle
                    await asyncio.sleep(2)

        except (websockets.exceptions.ConnectionClosedError, ConnectionRefusedError):
            # Sunucu çökerse veya bağlantı koparsa buraya düşer
            clear_output(wait=True)
            print("-------------------------------------------------------")
            print("🔌 SUNUCU BAĞLANTISI KOPTU VEYA SUNUCU KAPALI!")
            print("🔄 Yeniden bağlanılmaya çalışılıyor... (Lütfen bekleyin)")
            print("-------------------------------------------------------")
            await asyncio.sleep(3) # 3 saniye bekle ve en baştaki while True'ya dönüp tekrar dene
            
        except Exception as e:
            clear_output(wait=True)
            print(f"❌ Beklenmedik Hata: {e}")
            print("🔄 3 saniye sonra tekrar denenecek...")
            await asyncio.sleep(3)

# Dashboard'u başlat
await live_dashboard()

-------------------------------------------------------
🔌 SUNUCU BAĞLANTISI KOPTU VEYA SUNUCU KAPALI!
🔄 Yeniden bağlanılmaya çalışılıyor... (Lütfen bekleyin)
-------------------------------------------------------


CancelledError: 